In [1]:
#import statements
import os
import json
import torch
import numpy as np
import pandas as pd
import time
from torchvision import transforms
from scipy.cluster.hierarchy import fclusterdata

from deepproblog.model import Model
from deepproblog.engines import ExactEngine
from deepproblog.network import Network
from deepproblog.query import Query
from problog.logic import Term,Constant
from typing import Mapping

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

from classification.src.models import model as cls_model
from modelSSD import SSD300
from detect import detect
from utils import sort_bboxes, get_orientation, translate_labels, classification

#configs
images_folder      = r"C:\Users\jana\Pytorch rummick images\dataset\images"
ssd_model_path     = r"C:\Users\jana\Pytorch rummick images\model\tile_detection.pth"
color_model_path  = r"C:\Users\jana\Pytorch rummick images\classification\color_last.pth"
number_model_path = r"C:\Users\jana\Pytorch rummick images\classification\number_last.pth"
deepproblog_file  = r"rummikubexam.pl"
output_json_file  = "deepproblog_results.json"
checkpoint_file   = "checkpoint.json"   # lives in the same folder as this script

threshold_run = 0.10
threshold_set = 0.10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device used :",device)

#checkpoint helpers

def load_prev_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file,"r") as file:
            datas = json.load(file)
            done_images = set(datas.get("done_images",[]))
            results = datas.get("results",[])
            print(f"Checking checkpoints Resuming - {len(done_images)} images already processed,skipping them. ")
            return done_images,results
    print("Checkpoint not found - starting newly")
    return set(),[]

def saving_checkpoints(done_images,results):
    with open(checkpoint_file,"w") as write_file:
        json.dump({"done_images": list(done_images), "results" : results},write_file,indent=2)


class probcolornet(torch.nn.Module):
    def __init__(self, base_model, temperature=2.0):
        super().__init__()
        self.base = base_model
        self.temperature = temperature

    def forward(self,x):
        logits = self.base(x)/self.temperature
        probs = torch.softmax(logits,dim=1)
        return torch.clamp(probs,1e-6,1.0)
    
class mymapping(Mapping[Term,torch.Tensor]):
    def __init__(self):
        self.data = {}
    
    def __getitem__(self,key):
        if isinstance(key, tuple):
            key = key[0]
        if hasattr(key, "value"):
            key = key.value
        return self.data[key]
    
    def __iter__(self):
        return iter(self.data)
    
    def __len__(self):
        return len(self.data)
    
    def add(self,key,tensor):
        self.data[key] = tensor

def clustering_tiles(boundingboxes,relative_threshold=1.5):
    
    if len(boundingboxes) == 0:
        return {}
    
    centres = np.array([[(box[0]+box[2])/2 , (box[1]+box[3])/2] for box in boundingboxes])
    average_size = (np.mean([box[2]-box[0] for box in boundingboxes]) +
                    np.mean([box[3]-box[1] for box in boundingboxes])) /2
    label = fclusterdata(centres, t=average_size*relative_threshold ,
                         criterion="distance")
    
    clusters = {}
    for i,l in enumerate(label):
        clusters.setdefault(l-1,[]).append(boundingboxes[i])
    return clusters

def process_images(images_folder,ssd_model,color_model,number_model,logic_model):
    transform = transforms.Compose(
        [
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ]
    )

    all_files = sorted([file for file in os.listdir(images_folder)
                        if file.lower().endswith((".jpg",".png",".jpeg"))])
    
    done_images,results_summary = load_prev_checkpoint()
    remaining_images = [file for file in all_files if file not in done_images]

    print(f"Total images present : {len(all_files)}")
    print(f"Already processed_images : {len(done_images)}")
    print(f"To be Processed : {len(remaining_images)}\n")

    for image in remaining_images:
        image_path = os.path.join(images_folder,image)
        original_image = Image.open(image_path).convert("RGB")
        image_start = time.time()


        boundingboxes,_,_ = detect(
            original_image,
            ssd_model,
            min_score=0.7,
            max_overlap=0.3,
            top_k=200
        )

        clusters = clustering_tiles(boundingboxes)
        cluster_sorted = {}

        for i,boxes in clusters.items():
            orientation = get_orientation(boxes)
            cluster_sorted[i] = sort_bboxes(boxes,orientation)
        
        tile_mapping = mymapping()
        tile_tensors = {}

        for i,box in cluster_sorted.items():
            for j,bbox in enumerate(box):
                tile_image = original_image.crop(bbox)
                name = f"A{i}_{j}"
                transformed_tile = transform(tile_image)
                tile_tensors[name]=transformed_tile
                tile_mapping.add(name,transformed_tile)
        
        logic_model.add_tensor_source("tile_input",tile_mapping)

        all_run = []
        all_set = []
        n_valid_run = 0
        n_valid_set = 0

        for i , boxes in cluster_sorted.items():
            names = [f"A{i}_{x}" for x in range(len(boxes))]
            for nm in range(len(names)-2):
                triplets = [names[nm],names[nm+1],names[nm+2]]
                for j in triplets:
                    tile_tensor = tile_tensors[j].unsqueeze(0)
                    with torch.no_grad():
                        num = torch.softmax(number_model(tile_tensor)/2.0,dim=1)
                t1 = Term("tensor", Term("tile_input", Constant(triplets[0])))
                t2 = Term("tensor", Term("tile_input", Constant(triplets[1])))
                t3 = Term("tensor", Term("tile_input", Constant(triplets[2])))

                run_result = logic_model.solve([Query(Term("valid_run",t1,t2,t3))])[0]
                set_result = logic_model.solve([Query(Term("valid_set",t1,t2,t3))])[0]

                for k in run_result.result:
                    p_run = run_result.result[k].item()
                for k in set_result.result:
                    p_set = set_result.result[k].item()
                all_run.append(p_run)
                all_set.append(p_set)
                
                is_run = p_run > threshold_run and p_run >= p_set
                is_set = p_set > threshold_set and p_set >  p_run

                if is_run:
                    n_valid_run += 1
                if is_set:
                    n_valid_set += 1

                final_result = "Run" if is_run else ("Set" if is_set else "Neither Run nor Set")

                print(f"{final_result} run={p_run:.4f} set={p_set:.4f}"
                    f"({triplets[0]},{triplets[1]},{triplets[2]})")
        average_probability_run = float(np.mean(all_run)) if all_run else 0.0
        average_probability_set = float(np.mean(all_set)) if all_set else 0.0
        image_time = time.time() - image_start

        print(f"[{image}] valid_run : {n_valid_run}"
                f" : valid_set: {n_valid_set}"
                f" : time: {image_time:.3f} seconds")
        
        results_summary.append({
            "image":              image,
            "num_clusters":       len(cluster_sorted),
            "avg_valid_run_prob": average_probability_run,
            "avg_valid_set_prob": average_probability_set,
            "all_run_probs":      all_run,
            "all_set_probs":      all_set,
            "n_valid_run":        n_valid_run,
            "n_valid_set":        n_valid_set,
            "confidence":         max(average_probability_run, average_probability_set),
            "time_seconds":       round(image_time, 4),
        })



        done_images.add(image)
        saving_checkpoints(done_images,results_summary)

    return results_summary

def plot_results(results_summary):
    total_runs = sum(r["n_valid_run"] for r in results_summary)
    total_sets = sum(r["n_valid_set"] for r in results_summary)
    total      = total_runs + total_sets

    fig, ax = plt.subplots(figsize=(6, 5))

    bars = ax.bar(
        ["Valid Runs", "Valid Sets"],
        [total_runs, total_sets],
        color=["#378ADD", "#BA7517"],
        width=0.4
    )

    for bar, val in zip(bars, [total_runs, total_sets]):
        pct = f"{val / total * 100:.1f}%" if total > 0 else "0%"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{val}\n({pct})",
            ha="center", va="bottom", fontsize=13, fontweight="bold"
        )

    ax.set_ylabel("Count", fontsize=12)
    ax.set_title("Total valid run vs set triplets\nacross all images", fontsize=13)
    ax.set_ylim(0, max(total_runs, total_sets) * 1.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=12)

    plt.tight_layout()
    plt.savefig("dpl_counts.png", dpi=150, bbox_inches="tight")
    plt.close("all")
    print(f"Saved → dpl_counts.png  (runs={total_runs}, sets={total_sets})")
    
if __name__ == "__main__":

    ssd_model = SSD300(2, device)
    ssd_model.load_state_dict(torch.load(ssd_model_path, map_location=device))
    ssd_model.eval()

    color_model  = cls_model.get_model("resnet18", None, out_features=4)
    number_model = cls_model.get_model("resnet18", None, out_features=14)
    color_model.load_state_dict(torch.load(color_model_path,  map_location=device))
    number_model.load_state_dict(torch.load(number_model_path, map_location=device))
    color_model.eval()
    number_model.eval()  

    color_net  = Network(probcolornet(color_model),  "color_net",  batching=True)
    number_net = Network(probcolornet(number_model), "number_net", batching=True)

    logic_model = Model(deepproblog_file, [color_net, number_net])
    logic_model.set_engine(ExactEngine(logic_model), cache=True)

    total_start     = time.time()
    results_summary = process_images(
        images_folder, ssd_model, color_model, number_model, logic_model
    )

    total_time = time.time() - total_start

    full_total_time = round(sum(r["time_seconds"] for r in results_summary), 4)

    print(f"DeepProbLog total inference time: {full_total_time:.2f}s")

    plot_results(results_summary)
    
    output_json = {
        "total_time": round(full_total_time, 4),
        "results":    results_summary,
    }
    with open(output_json_file, "w") as f:
        json.dump(output_json, f, indent=2)
    print(f"Saved in {output_json_file}")
  
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print(f"Deleted checkpoint ({checkpoint_file}) — all done!")



c:\Users\jana\AppData\Local\Programs\Python\Python310\lib\site-packages\deepproblog\engines\__init__.py:6: UserWarning: ApproximateEngine is not available as PySwip could not be found
  warnings.warn("ApproximateEngine is not available as PySwip could not be found")


Device used : cuda
using pretrained weights

Loaded base model.



C:\Users\jana\AppData\Local\Temp\ipykernel_11064\952346667.py:264: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ssd_model.load_state_dict(torch.load(ssd_model_path, map_loc

Caching ACs
Checking checkpoints Resuming - 216 images already processed,skipping them. 
Total images present : 285
Already processed_images : 216
To be Processed : 69

Run run=0.5593 set=0.0000(A1_0,A1_1,A1_2)
Run run=0.2989 set=0.0000(A2_0,A2_1,A2_2)
Neither Run nor Set run=0.0066 set=0.0008(A2_1,A2_2,A2_3)
Neither Run nor Set run=0.0000 set=0.0001(A2_2,A2_3,A2_4)
Neither Run nor Set run=0.0001 set=0.0001(A2_3,A2_4,A2_5)
Neither Run nor Set run=0.0000 set=0.0000(A2_4,A2_5,A2_6)
Neither Run nor Set run=0.0001 set=0.0001(A2_5,A2_6,A2_7)
Neither Run nor Set run=0.0110 set=0.0520(A0_0,A0_1,A0_2)
[BL3_9240.JPG] valid_run : 2 : valid_set: 0 : time: 28.203 seconds
Neither Run nor Set run=0.0029 set=0.0181(A2_0,A2_1,A2_2)
Neither Run nor Set run=0.0004 set=0.0001(A0_0,A0_1,A0_2)
Neither Run nor Set run=0.0000 set=0.0001(A1_0,A1_1,A1_2)
[BL3_9241.JPG] valid_run : 0 : valid_set: 0 : time: 2.549 seconds
Neither Run nor Set run=0.0003 set=0.0000(A0_0,A0_1,A0_2)
Neither Run nor Set run=0.0001 set